**Objective**

Extract standardized image features from the sampled DMS dataset.

Input: data/samples/
Output: data/processed/dms_features.csv

In [1]:
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

In [2]:
PROJECT_DIR = Path("/workspaces/DBA-ai-trust-score-framework-Interim")

SAMPLES_DIR = PROJECT_DIR / "data" / "samples"

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
metadata = pd.read_csv(
    PROCESSED_DIR / "dms_cleaned.csv"
)

metadata.head()

,file_name,label,dataset,original_path
0,gB_7_s2_2019-03-11T14-12-25-01-00_ir_face_mp4-...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...
1,gA_4_s2_ir_face_mp4-372_jpg.rf.448fec2e90c845f...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...
2,gA_5_s2_ir_face_mp4-130_jpg.rf.1081aec5f8fd8da...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...
3,gB_9_s2_2019-03-07T16-21-20-01-00_ir_face_mp4-...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...
4,gA_5_s2_ir_face_mp4-246_jpg.rf.f7bcf481f361e68...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...


In [4]:
def extract_features(image_path):

    image = cv2.imread(str(image_path))

    if image is None:
        return None

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    height, width = gray.shape

    image_area = width * height

    brightness = np.mean(gray)

    contrast = np.std(gray)

    blur_score = cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

    histogram = cv2.calcHist(
        [gray],
        [0],
        None,
        [256],
        [0,256]
    )

    histogram = histogram / histogram.sum()

    entropy = -np.sum(
        histogram * np.log2(histogram + 1e-10)
    )

    edges = cv2.Canny(
        gray,
        100,
        200
    )

    edge_density = np.mean(edges > 0)

    size_mb = image_path.stat().st_size / (1024*1024)

    return {

        "width": width,

        "height": height,

        "image_area": image_area,

        "aspect_ratio": width / height,

        "size_mb": size_mb,

        "size_per_pixel": size_mb / image_area,

        "brightness": brightness,

        "contrast": contrast,

        "blur_score": blur_score,

        "entropy": entropy,

        "edge_density": edge_density,

        "is_square": int(
            abs(width-height) <= 10
        )

    }

In [5]:
feature_rows = []

for _, row in tqdm(
    metadata.iterrows(),
    total=len(metadata)
):

    image_path = (

        SAMPLES_DIR /

        row["label"] /

        row["file_name"]

    )

    if not image_path.exists():
        continue

    features = extract_features(image_path)

    if features is None:
        continue

    feature_rows.append({

        **row.to_dict(),

        **features

    })

100%|██████████| 1400/1400 [00:11<00:00, 121.13it/s]


In [6]:
feature_dataset = pd.DataFrame(
    feature_rows
)

print(feature_dataset.shape)

feature_dataset.head()

(1000, 16)


,file_name,label,dataset,original_path,width,height,image_area,aspect_ratio,size_mb,size_per_pixel,brightness,contrast,blur_score,entropy,edge_density,is_square
0,gB_7_s2_2019-03-11T14-12-25-01-00_ir_face_mp4-...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...,561,640,359040,0.876563,0.024924,6.941922e-08,54.612954,35.971797,16.303268,6.854137,0.007910,0
1,gA_4_s2_ir_face_mp4-372_jpg.rf.448fec2e90c845f...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...,539,640,344960,0.842187,0.023157,6.712987e-08,66.157813,52.975199,30.971761,7.077067,0.011138,0
2,gA_5_s2_ir_face_mp4-130_jpg.rf.1081aec5f8fd8da...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...,517,640,330880,0.807813,0.024449,7.389189e-08,85.879162,59.912889,22.451453,7.373963,0.011424,0
3,gB_9_s2_2019-03-07T16-21-20-01-00_ir_face_mp4-...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...,640,635,406400,1.007874,0.028551,7.025370e-08,85.189166,71.628915,22.311506,7.335597,0.010723,1
4,gA_5_s2_ir_face_mp4-246_jpg.rf.f7bcf481f361e68...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...,509,640,325760,0.795312,0.023486,7.209644e-08,91.038258,68.262635,20.952316,7.441490,0.009486,0


In [7]:
feature_dataset.isnull().sum()

file_name         0
label             0
dataset           0
original_path     0
width             0
height            0
image_area        0
aspect_ratio      0
size_mb           0
size_per_pixel    0
brightness        0
contrast          0
blur_score        0
entropy           0
edge_density      0
is_square         0
dtype: int64

In [8]:
feature_dataset.describe().T

,count,mean,std,min,25%,50%,75%,max
width,1000.0,5.943660e+02,7.222282e+01,3.840000e+02,5.450000e+02,6.400000e+02,6.400000e+02,683.000000
height,1000.0,5.997560e+02,6.813045e+01,3.620000e+02,5.680000e+02,6.400000e+02,6.400000e+02,640.000000
image_area,1000.0,3.546171e+05,5.018300e+04,2.316800e+05,3.072000e+05,3.539200e+05,4.096000e+05,409600.000000
aspect_ratio,1000.0,1.012229e+00,2.140544e-01,6.000000e-01,8.515625e-01,1.000000e+00,1.126771e+00,1.778646
size_mb,1000.0,3.835767e-02,2.695190e-02,7.940292e-03,2.176523e-02,3.739834e-02,4.543734e-02,0.341972
size_per_pixel,1000.0,1.063689e-07,7.674091e-08,3.283704e-08,6.506241e-08,9.882497e-08,1.231713e-07,0.000001
brightness,1000.0,9.395080e+01,2.197073e+01,2.003616e+01,8.371009e+01,9.416125e+01,1.061456e+02,161.999124
contrast,1000.0,7.012734e+01,1.545277e+01,1.074828e+01,6.396478e+01,7.470226e+01,8.042018e+01,97.716413
blur_score,1000.0,1.027319e+03,3.177544e+03,1.978707e+00,1.374295e+01,3.049142e+02,5.409008e+02,23670.466941
entropy,1000.0,7.318608e+00,3.849283e-01,5.100131e+00,7.120730e+00,7.425938e+00,7.603230e+00,7.934317


In [9]:
feature_dataset.to_csv(

    PROCESSED_DIR /

    "dms_features.csv",

    index=False

)

print("✅ dms_features.csv saved")

✅ dms_features.csv saved


In [10]:
print(feature_dataset.info())

print()

print(feature_dataset.head())

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   file_name       1000 non-null   str    
 1   label           1000 non-null   str    
 2   dataset         1000 non-null   str    
 3   original_path   1000 non-null   str    
 4   width           1000 non-null   int64  
 5   height          1000 non-null   int64  
 6   image_area      1000 non-null   int64  
 7   aspect_ratio    1000 non-null   float64
 8   size_mb         1000 non-null   float64
 9   size_per_pixel  1000 non-null   float64
 10  brightness      1000 non-null   float64
 11  contrast        1000 non-null   float64
 12  blur_score      1000 non-null   float64
 13  entropy         1000 non-null   float32
 14  edge_density    1000 non-null   float64
 15  is_square       1000 non-null   int64  
dtypes: float32(1), float64(7), int64(4), str(4)
memory usage: 298.3 KB
None

                 

In [11]:
dynamic_range = gray.max() - gray.min()

NameError: name 'gray' is not defined